**Сама реализация LDA**

убираем слова, которые не несут смысла (MEANLESS_WORDS) - артикли, предлоги, союзы и т.п.

In [58]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.datasets import fetch_20newsgroups
import warnings
warnings.filterwarnings('ignore')
EXTENDED_STOP_WORDS = set(ENGLISH_STOP_WORDS) | {
    # Очень частые слова в newsgroups
    'use', 'used', 'using', 'way', 'time', 'thing', 'things',
    'know', 'make', 'made', 'making', 'get', 'got', 'getting',
    'people', 'think', 'thinking', 'say', 'said', 'saying',
    'really', 'just', 'want', 'like', 'course', 'case',
    'new', 'available', 'probably', 'better', 'does',
    'day', 'days', 'year', 'years',
    'small', 'going', 'good', 'bad', 'work', 'works',
    'high', 'higher', 'low', 'single', 'option', 'options',
    'number', 'numbers', 'car', 'cars', 'bit', 'bits',
    'post', 'posts', 'add', 'come', 'comes',
    'old', 'pay', 'control', 'power', 'speed', 'auto',
    'team', 'board', 'mac', 'capability', 'starters',
    'insurance', 'scsi', 'don', 'sure',
    
    # Домены и технические
    'edu', 'com', 'org', 'edu', 'net', 'url', 'http', 'ftp',
    
    # Междометия и наречия
    'thanks', 'thank', 'quite', 'maybe', 'perhaps', 'probably',
    'actually', 'really', 'very', 'definitely', 'certainly',
    
    # Невнятные слова
    'hand', 'left', 'right', 'point', 'bit', 'head', 'face',
    'back', 'side', 'top', 'bottom', 'front', 'middle', 'end',
    'start', 'begin', 'happen', 'happened', 'happening',
    
    # Служебные слова
    'info', 'data', 'list', 'email', 'message', 'post', 'posts',
    'file', 'files', 'line', 'lines', 'item', 'items',
    'subject', 'wrote', 'reply', 'comment', 'comments',
    
    # Общие слова
    'great', 'good', 'bad', 'different', 'possible', 'true',
    'false', 'help', 'helped', 'helping', 'need', 'needed',
    'called', 'called', 'run', 'runs', 'running',
    'cut', 'copy', 'move', 'delete', 'create', 'created',
    
    # Глаголы состояния
    'hope', 'hoped', 'hoping', 'seem', 'seemed', 'appears',
    'appears', 'feel', 'felt', 'feeling', 'mean', 'meant',
    
    # Остаток
    'currently', 'understand', 'understand', 'heard', 'faster',
    'supply', 'required', 'required', 'defense', 'environment',
    'systems', 'systems', 'war', 'little', 'jews',
    'california', 'personal', 'class', 'cost', 'buy',
    'gun', 'accidents', 'deductible', 'mention', 'mentioned'
}

stop_words = list(EXTENDED_STOP_WORDS)

class LatentDirichletAllocationWithVectorizer:
    def __init__(self, n_topics=5, alpha=0.1, beta=0.01, iterations=50):
        self.n_topics = n_topics
        self.alpha = alpha
        self.beta = beta
        self.iterations = iterations
        
        self.vectorizer = None
        self.vocab = None
        self.id2word = None
        self.vocab_size = None
        
        self.doc_term_matrix = None
        self.n_docs = None
        self.n_words_total = None
        
        self.n_k_w = None
        self.n_d_k = None
        self.n_k = None
        
        self.word_assignments = None
        self.word_ids = None
        self.doc_ids = None
    
    def fit(self, texts, max_features=1000, min_df=3, max_df=0.8):
        self.vectorizer = CountVectorizer(
            max_features=max_features,
            min_df=min_df,
            max_df=max_df,
            stop_words=stop_words,
            lowercase=True,
            token_pattern=r'[a-z]{4,}',
            analyzer = 'word'
        )
        
        self.doc_term_matrix = self.vectorizer.fit_transform(texts)
        
        self.vocab = self.vectorizer.get_feature_names_out()
        self.vocab_size = len(self.vocab)
        self.id2word = {idx: word for idx, word in enumerate(self.vocab)}
        self.n_docs = self.doc_term_matrix.shape[0]
        self.n_words_total = self.doc_term_matrix.sum()
        
        self._matrix_to_word_list()
        
        print(f"Total words to process: {len(self.word_ids)}")
        
        self._initialize_counters()
        self._gibbs_sampling()
        
        return self
    
    def _matrix_to_word_list(self):
        word_ids = []
        doc_ids = []
        
        doc_term_coo = self.doc_term_matrix.tocoo()
        
        # Итерировать по ненулевым элементам
        for doc_idx, word_idx, count in zip(doc_term_coo.row, doc_term_coo.col, doc_term_coo.data):
            for _ in range(int(count)):
                doc_ids.append(doc_idx)
                word_ids.append(word_idx)
        
        self.word_ids = np.array(word_ids, dtype=np.int32)
        self.doc_ids = np.array(doc_ids, dtype=np.int32)
    
    def _initialize_counters(self):
        print("Initialising counters")
        
        self.n_k_w = np.zeros((self.n_topics, self.vocab_size), dtype=np.int32)
        self.n_d_k = np.zeros((self.n_docs, self.n_topics), dtype=np.int32)
        self.n_k = np.zeros(self.n_topics, dtype=np.int32)
        
        self.word_assignments = np.random.randint(0, self.n_topics, len(self.word_ids))
        
        for i in range(len(self.word_ids)):
            doc_id = self.doc_ids[i]
            word_id = self.word_ids[i]
            topic = self.word_assignments[i]
            
            self.n_k_w[topic, word_id] += 1
            self.n_d_k[doc_id, topic] += 1
            self.n_k[topic] += 1
    
    def _gibbs_sampling(self):
        V = self.vocab_size
        
        print("Gibbs sampling")
        
        for iteration in range(self.iterations):
            for i in range(len(self.word_ids)):
                doc_id = self.doc_ids[i]
                word_id = self.word_ids[i]
                old_topic = self.word_assignments[i]
                
                self.n_d_k[doc_id, old_topic] -= 1
                self.n_k_w[old_topic, word_id] -= 1
                self.n_k[old_topic] -= 1
                
                p_k = np.zeros(self.n_topics, dtype=np.float64)
                
                for k in range(self.n_topics):
                    numerator = (self.n_d_k[doc_id, k] + self.alpha) * \
                               (self.n_k_w[k, word_id] + self.beta)
                    denominator = self.n_k[k] + V * self.beta
                    p_k[k] = numerator / denominator
                
                p_k_sum = p_k.sum()
                if p_k_sum > 0:
                    p_k /= p_k_sum
                else:
                    p_k = np.ones(self.n_topics) / self.n_topics
                
                new_topic = np.random.choice(self.n_topics, p=p_k)
                self.word_assignments[i] = new_topic
                
                self.n_d_k[doc_id, new_topic] += 1
                self.n_k_w[new_topic, word_id] += 1
                self.n_k[new_topic] += 1
            
            if (iteration + 1) % max(1, self.iterations // 5) == 0:
                print(f"  Iteration {iteration + 1}/{self.iterations}")
        
        print("Sampling complete")
        
    def get_topics(self, top_n_words=10):
        topics = {}
        for topic_id in range(self.n_topics):
            top_word_indices = np.argsort(self.n_k_w[topic_id, :])[-top_n_words:][::-1]
            top_words = [self.vocab[idx] for idx in top_word_indices]
            topics[topic_id] = top_words
        return topics
    
    def get_document_topics(self, doc_id):
        doc_topics = {}
        total_words = self.n_d_k[doc_id, :].sum()
        
        if total_words > 0:
            for topic_id in range(self.n_topics):
                doc_topics[topic_id] = self.n_d_k[doc_id, topic_id] / total_words
        else:
            for topic_id in range(self.n_topics):
                doc_topics[topic_id] = 1.0 / self.n_topics
        
        return doc_topics
    
    def print_topics(self, top_n_words=10):
        topics = self.get_topics(top_n_words)
        for topic_id, words in topics.items():
            print(f"\nTopic {topic_id + 1}:")
            print(f"  {', '.join(words)}")
    
    def print_document_topics(self, documents=None, num_docs=3):
        
        for doc_idx in range(min(num_docs, self.n_docs)):
            if documents is not None:
                if hasattr(documents, 'data'):
                    doc_text = documents.data[doc_idx][:60]
                else:
                    doc_text = documents[doc_idx][:60]
            else:
                doc_text = f"Document {doc_idx}"
            
            print(f"\nDocument {doc_idx + 1}: '{doc_text}...'")
            doc_topics = self.get_document_topics(doc_idx)
            
            sorted_topics = sorted(doc_topics.items(), 
                                  key=lambda x: x[1],
                                  reverse=True)
            
            for topic_id, prob in sorted_topics[:3]:
                print(f"  Topic {topic_id + 1}: {prob:.2%}")
    
    def print_statistics(self):
        print(f"Documents: {self.n_docs}")
        print(f"Vocabulary size: {self.vocab_size} unique words")
        print(f"Total word occurrences: {len(self.word_ids)}")
        print(f"Topics: {self.n_topics}")
        print(f"Iterations: {self.iterations}")
        print(f"Alpha (doc-topic): {self.alpha}")
        print(f"Beta (topic-word): {self.beta}")

def main():
    newsgroups = fetch_20newsgroups(
        subset='train',
        remove=('headers', 'footers', 'quotes')
    )
    
    texts = newsgroups.data[:100]  # 100 документов
    
    lda = LatentDirichletAllocationWithVectorizer(
        n_topics=5,
        alpha=0.1,
        beta=0.01,
        iterations=20
    )
    
    lda.fit(
    texts,
    max_features=500,
    min_df=3,
    max_df=0.5,)

    
    lda.print_topics(top_n_words=10)
    lda.print_document_topics(newsgroups, num_docs=3)
    lda.print_statistics()
    
if __name__ == "__main__":
    main()


Total words to process: 1953
Initialising counters
Gibbs sampling
  Iteration 4/20
  Iteration 8/20
  Iteration 12/20
  Iteration 16/20
  Iteration 20/20
Sampling complete

Topic 1:
  windows, program, record, image, assist, software, technology, images, application, division

Topic 2:
  army, population, religion, large, contact, computer, driven, road, book, religious

Topic 3:
  space, problem, chip, uses, memory, fact, information, problems, close, situation

Topic 4:
  armenian, armenians, world, government, understanding, provide, phone, mode, soviet, source

Topic 5:
  state, doesn, game, miles, tried, trying, request, response, failed, university

Document 1: 'I was wondering if anyone out there could enlighten me on th...'
  Topic 5: 75.00%
  Topic 3: 25.00%
  Topic 1: 0.00%

Document 2: 'A fair number of brave souls who upgraded their SI clock osc...'
  Topic 2: 85.71%
  Topic 5: 14.29%
  Topic 1: 0.00%

Document 3: 'well folks, my mac plus finally gave up the ghost this week

провели запуск при альфа = 0.1 и бета = 0.01

альфа отвечает за плотность топиков в тексте

бета отвечает за количество слов в топиках

параметры и список ненужных слов подобраны опытным путем - я перебирал варианты

**Результат**

первый - с компьютерными технологиями

второй - военный конфликт, страны

третий - космические технологии

четвертый - что то с Арменией

пятый - уже что то невнятное